In [1]:
2+2

4

In [2]:
import instructor
from google import genai
from pydantic import BaseModel,Field

In [3]:
import os
from dotenv import  load_dotenv

In [4]:
load_dotenv() #Once load_dotenv() runs, genai.Client() automatically detects and reads GEMINI_API_KEY from your environment without needing to pass it manually:

True

In [5]:
api_key=os.getenv("GEMINI_API_KEY")


In [6]:
#Initialize Google GenAI client

client=genai.Client(api_key=api_key)

In [7]:

# We define the raw text we want to process
corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")


=== STARTING PROMPT ENGINEERING PIPELINE ===



In [8]:
# TECHNIQUE 1: ZERO-SHOT PROMPTING

print("1. ZERO-SHOT OUTPUT:")
zero_shot_prompt = f"Extract the author from this text: {corporate_text}"
response_zero = client.models.generate_content(
    model='gemini-3.6-flash', 
    contents=zero_shot_prompt
)
print(response_zero.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


1. ZERO-SHOT OUTPUT:
The author of the text is **Jaymin**.


In [ ]:
# TECHNIQUE 2: FEW-SHOT PROMPTING
# ---------------------------------------------------------
print("2. FEW-SHOT OUTPUT:")
few_shot_prompt = f"""
Extract the author from the text. Respond ONLY with the name.
Example 1 Input: "Memo: Server down. Written by Tony." -> Example 1 Output: Tony
Example 2 Input: "Update from Marten regarding sales." -> Example 2 Output: Marten
Real Input: {corporate_text}
"""

response_few = client.models.generate_content(
    model='gemini-3.6-flash', 
    contents=few_shot_prompt
)
print(response_few.text)


In [ ]:
print("3. CHAIN-OF-THOUGHT OUTPUT:")
cot_prompt = f"""
Analyze the following text to find the primary author.
Think step-by-step. First, identify any names in the text. Second, look for
keywords like 'Written by' or 'From' near those names. Third, declare the author.
Text: {corporate_text}
"""

response_cot=client.models.generate_content(
    model="gemini-3.8-flash",
    contents=cot_prompt
)

print(response_cot)

In [20]:
class DocumentMetadata(BaseModel):
    title: str = Field(description="The official title of the document.")
    author: str = Field(description="The name of the employee who wrote it. Use 'UNKNOWN' if missing.")
    summary: str = Field(description="A strict 1-sentence summary of the content.")

mega_prompt = f"""
You are an expert corporate data extraction algorithm.
CONTEXT: Processing raw text extracted from corporate documents.
TASK: Extract the title, author, and a 1-sentence summary based on the provided schema.
RULES: Do not guess. If data is missing, follow the schema instructions exactly.

corporate documents:
{corporate_text}
"""
response_structure=client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=mega_prompt,
    config={
        "response_mime_type":"application/json",
        "response_schema":DocumentMetadata
    }
)

print(response_structure.text)

{"title": "MEMO RE: Q4 Scaling", "author": "Jaymin", "summary": "The organization needs to increase the cloud budget for the Extract Once, Judge Many pipeline due to current architectural bottlenecks."}


In [21]:
extracted_data=DocumentMetadata.model_validate_json(response_structure.text)

In [22]:
print(extracted_data)


title='MEMO RE: Q4 Scaling' author='Jaymin' summary='The organization needs to increase the cloud budget for the Extract Once, Judge Many pipeline due to current architectural bottlenecks.'


In [23]:
print(extracted_data.author)

Jaymin


In [8]:
# TECHNIQUE 4: MEGA-PROMPT + STRUCTURED JSON (ENTERPRISE GRADE)

print("4. STRUCTURED JSON OUTPUT (PYDANTIC + INSTRUCTOR):")

# A. Define the strict Pydantic Data Blueprint
class DocumentMetadata(BaseModel):
    title: str = Field(description="The official title of the document.")
    author: str = Field(description="The name of the employee who wrote it. Use 'UNKNOWN' if missing.")
    summary: str = Field(description="A strict 1-sentence summary of the content.")

# B. Patch the Gemini client with Instructor to unlock strict JSON mode,forces the llm to obey the schema
instructor_client = instructor.from_genai(
    client=client,
    mode=instructor.Mode.JSON
)

# C. Craft the Mega-Prompt
mega_prompt = f"""
You are an expert corporate data extraction algorithm.
CONTEXT: Processing raw text extracted from corporate documents.
TASK: Extract the title, author, and a 1-sentence summary based on the provided schema.
RULES: Do not guess. If data is missing, follow the schema instructions exactly.

corporate documents:
{corporate_text}
"""

# D. Execute the API Call using response_model
extracted_data = instructor_client.chat.completions.create(
    model='gemini-3.1-flash-lite',
    messages=[{"role": "user", "content": mega_prompt}],
    response_model=DocumentMetadata # Forcing the AI to obey our Pydantic class
)

# E. The Result is now a Python Object, not a string!
print(extracted_data.model_dump_json(indent=2))
print("\n[System] Successfully extracted author directly into Python variable:", extracted_data.author)



4. STRUCTURED JSON OUTPUT (PYDANTIC + INSTRUCTOR):


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{
  "title": "MEMO RE: Q4 Scaling",
  "author": "Jaymin",
  "summary": "The organization requires increased cloud budget to address architectural bottlenecks in the 'Extract Once, Judge Many' pipeline."
}

[System] Successfully extracted author directly into Python variable: Jaymin


In [18]:

#zero short

In [ ]:
import instructor
from google import genai
from pydantic import BaseModel,Field

import os
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")

client=genai.Client(api_key=api_key)

corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")


print("*"*40)

print("zero short prompting")

zero_short_prompt=f"extract the author from the text:{corporate_text}"

response=client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=zero_short_prompt

)

print(response.text)


In [19]:
#few short

In [ ]:
import instructor
from google import genai

import os
from dotenv import load_dotenv

load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")

corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")

print("*"*80)

client=genai.Client(api_key=api_key)

few_short_prompt=f""" Extract the author from the context real input :{corporate_text}.
  Based on examples:
  harry potter:jk rowling
  alchemist:paulo chelo
  """

response=client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=few_short_prompt
)

print(response.text)

In [20]:
#cot

In [15]:
import instructor
from google import genai

from dotenv import load_dotenv

import os

load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")

corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")


client=genai.Client(api_key=api_key)

chain_of_thought_prompt=f"""
Alway think step by step before answering a questions based on context :
{corporate_text}.
Extract the author details from the given context.
"""

response=client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=chain_of_thought_prompt,

)

print(response.text)


=== STARTING PROMPT ENGINEERING PIPELINE ===

To extract the author details from the given context, let's follow these steps:

1. **Scan the text** for labels or keywords related to authorship, such as "Written by", "Author", or names at the top/bottom of the memo.
2. **Locate the relevant line:** The memo contains the line "Written by: Jaymin".
3. **Identify the author:** Based on that line, the author of the memo is Jaymin.

**Answer:**
Jaymin


#pydantic

In [ ]:
class DocumentMetadata(BaseModel):
    title: str = Field(description="The official title of the document.")
    author: str = Field(description="The name of the employee who wrote it. Use 'UNKNOWN' if missing.")
    summary: str = Field(description="A strict 1-sentence summary of the content.")

In [40]:
import instructor
from google import genai
from dotenv import load_dotenv
from pydantic import BaseModel,Field
import os
load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")

corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")

class DocumentMetadata(BaseModel):
    title:str=Field(description="the title of the document")
    author:str=Field(description="the name of the employee who wrote it.Use Unknown if missing")
    summary:int=Field(description="a strict one sentence of the content")

mega_prompt = f"""
You are an expert corporate data extraction algorithm.
TASK: Extract the title,author and a 1-sentence summary based on the provided schema.
RULES: Do not guess. If data is missing, follow the schema instructions exactly.

context:
{corporate_text}
"""

client=genai.Client(api_key=api_key)

response=client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=mega_prompt,
    config={
        "response_mime_type":"application/json",
        "response_schema":DocumentMetadata
    }

)

print("*"*100)
print(response.text)


=== STARTING PROMPT ENGINEERING PIPELINE ===

****************************************************************************************************
{
  "title": "Q4 Scaling",
  "author": "Jaymin",
  "summary": 1
}


In [28]:
#pydantic +instructot


In [44]:
import instructor
from google import genai
from pydantic import BaseModel,Field
from dotenv import load_dotenv

import os
load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")

# B. Patch the Gemini client with Instructor to unlock strict JSON mode,forces the llm to obey the schema
instructor_client = instructor.from_genai(
    client=client,
    mode=instructor.Mode.JSON
)

client=genai.Client(api_key=api_key)

instructor_client=instructor.from_genai(
    client=client,
    mode=instructor.Mode.JSON
)


corporate_text = """
MEMO RE: Q4 Scaling
Date: Oct 12, 2026
Written by: Jaymin
We need more cloud budget for the 'Extract Once, Judge Many' pipeline.
The architecture is bottlenecking. Tony and Marten will lead the review.
"""

print("=== STARTING PROMPT ENGINEERING PIPELINE ===\n")


class DocumentMetadata(BaseModel):
    title:str=Field(description="the title of the document")
    author:str=Field(description="the author of the document")
    summary:str=Field(description=" summary of the document")
    words:int=Field(description="number of words in the summary")


mega_prompt = f"""
You are an expert corporate data extraction algorithm.
CONTEXT: Processing raw text extracted from corporate documents.
TASK: Extract the title, author ,a 1-sentence summary and word number in the summary sentence based on the provided schema.
RULES: Do not guess. If data is missing, follow the schema instructions exactly.

corporate documents:
{corporate_text}
"""

# response=client.models.generate_content(
#     model="gemini-3.5-flash-lite",
#     contents=mega_prompt,
#     config={
#         "response_schema":DocumentMetadata,
#         "response_mimi_type":"application/json"
#     }
# )

# # D. Execute the API Call using response_model
# extracted_data = instructor_client.chat.completions.create(
#     model='gemini-3.1-flash-lite',
#     messages=[{"role": "user", "content": mega_prompt}],
#     response_model=DocumentMetadata # Forcing the AI to obey our Pydantic class
# )

# # E. The Result is now a Python Object, not a string!
# print(extracted_data.model_dump_json(indent=2))
# print("\n[System] Successfully extracted author directly into Python variable:", extracted_data.author)

response=instructor_client.chat.completions.create(
    model="gemini-3.5-flash-lite",
    messages=[{"role":"user","content":mega_prompt}],
    response_model=DocumentMetadata
)
print(response) #object
print(response.model_dump_json(indent=2)) #2 space after {}



=== STARTING PROMPT ENGINEERING PIPELINE ===

title='MEMO RE: Q4 Scaling' author='Jaymin' summary='Jaymin requests additional cloud budget to resolve architecture bottlenecks in the Extract Once, Judge Many pipeline, with Tony and Marten leading the review.' words=23
{
  "title": "MEMO RE: Q4 Scaling",
  "author": "Jaymin",
  "summary": "Jaymin requests additional cloud budget to resolve architecture bottlenecks in the Extract Once, Judge Many pipeline, with Tony and Marten leading the review.",
  "words": 23
}


In [ ]:
import instructor
from google import  genai

from pydantic import BaseModel,Field
from dotenv import load_dotenv

import os

load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")


client=genai.Client(api_key=api_key)

class DocumentMetadata(BaseModel):
    titel:str=Field(description="the title of the document")
    author:str=Field(description="the author written by")


response=client.models.generate_content(
    model="",
    contents=mega_prompt,
    config={
        "response_mimi_type":"application/json",
        "response_schema":DocumentMetadata
    }
)

client=genai.Client(api_key=api_key)

instructor_client=instructor.from_genai(
    clinet=client,
    mode=instructor.mode.json
)

response=instructor_client.chat.completion.create(
    model="",
    messages=[{"role":"user","content":mega_prompt}],
    response_model=DocumentMetadata

)

response.model_dump_json(indent=2)
